# Day 5 Content

In [13]:
import  json

from urllib3.util import wait

from modules.day02_conditions_loops import last_error
from modules.day04_file_io_json import result, cleaned

In [4]:
import json

from notebooks.Day05_Exception_Handling_Logging import bad_json, bad_type, response


def safe_to_int(value):
    try:
        return int(value)
    except ValueError:
        print(f" ValueError: '{value}' cannot be converted to int")
        return None

print(safe_to_int('1001'))
print(safe_to_int('abc'))
print(safe_to_int('3.14'))

1001
 ValueError: 'abc' cannot be converted to int
None
 ValueError: '3.14' cannot be converted to int
None


In [6]:
def get_customer_name(customer : dict, key : str):
    try:
        return customer[key].strip().title()
    except KeyError:
        print(f"  KeyError: key '{key}' not found in customer dict")
        return 'UnKnown'
    except AttributeError:
        print(f"  AttributeError: value for '{key}' is None")
        return 'UnKnown'

customer = {'customer_id': '1001', 'name': 'Yogavardhan Reddy', 'email' : None}

print(get_customer_name(customer, 'name'))
print(get_customer_name(customer, 'email'))
print(get_customer_name(customer, 'city'))

Yogavardhan Reddy
  AttributeError: value for 'email' is None
UnKnown
  KeyError: key 'city' not found in customer dict
UnKnown


In [14]:
def parse_csv_file(value: str, target_type: type):
    try:
        return target_type(value)
    except (ValueError,TypeError):
        print(f"  Could not convert '{value}' to {target_type.__name__}")
        return None

print(parse_csv_file('205.21',  float))
print(parse_csv_file('abc',     float))
print(parse_csv_file(None,      float))

205.21
  Could not convert 'abc' to float
None
  Could not convert 'None' to float
None


In [15]:
bad_json = '{"category": "TRACK_ORDER", missing_quote: "high"}'

try:
    parsed = json.loads(bad_json)
except json.JSONDecodeError as e:
    print(f"JSONDecodeError: {e}")
    print(f"    message : {e.msg}")
    print(f"    position: line {e.lineno}, column {e.colno}")
    parsed = {}

print(f"parsed  =  {parsed}")

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 29 (char 28)
    message : Expecting property name enclosed in double quotes
    position: line 1, column 29
parsed  =  {}


In [21]:
import time

def parse_with_timing(raw_response: str) -> dict:

    start = time.perf_counter()
    result = {}

    try:
        cleaned = raw_response.strip()
        if cleaned.startswith('```'):
            cleaned = cleaned.split('\n',1)[-1].rsplit('```',1)[0].strip()
        result = json.loads(cleaned)
        print(f"  Parse Succeeded: {result}")

    except json.JSONDecodeError as e:
        print(f"  Parse FAILED:  {e.msg}")
        result = {}

    finally:
        elapsed_ms = round((time.perf_counter() - start) * 1000,2)
        print(f" Elapsed:  {elapsed_ms}ms (finally block- always runs")

    return result

# Test 1 Valid JSON
print("Valid JSON:")
parse_with_timing('{"category": "TRACK_ORDER", "confidence": "high"}')
print()

# Test 2 invalid JSON
print("Invalid JSON:")
parse_with_timing('{bad json here}')

Valid JSON:
  Parse Succeeded: {'category': 'TRACK_ORDER', 'confidence': 'high'}
 Elapsed:  0.13ms (finally block- always runs

Invalid JSON:
  Parse FAILED:  Expecting property name enclosed in double quotes
 Elapsed:  0.1ms (finally block- always runs


{}

In [24]:
def validate_customer_id(customer_id: str) -> int:
    try:
        cid = int(customer_id)
    except ValueError as e:
        raise ValueError(f"customer_id must be a number, got: '{customer_id}'") from e

    if cid <= 0:
        raise ValueError(f"customer_id must be positive, got: {cid}")
    return cid

#Test: Valid id
print(validate_customer_id('1001'))

#Test: invalid id
try:
    validate_customer_id("abc")
except ValueError as e:
    print(f"Cought: {e}")

#Test: negative id
try:
    validate_customer_id('-5')
except ValueError as e:
    print(f"Cought: {e}")

1001
Cought: customer_id must be a number, got: 'abc'
Cought: customer_id must be positive, got: -5


In [27]:
import logging

logging.basicConfig(
    level= logging.DEBUG,
    format='%(levelname)-8s %(message)s',
    force=True,
)

log = logging.getLogger(__name__)
log.debug(   'DEBUG   — detailed info: only shown in development')
log.info(    'INFO    — normal operation: function started, request received')
log.warning( 'WARNING — unexpected but handled: missing CSV column')
log.error(   'ERROR   — something failed: json.JSONDecodeError caught')
log.critical('CRITICAL— serious failure: database unreachable')

DEBUG    DEBUG   — detailed info: only shown in development
INFO     INFO    — normal operation: function started, request received
WARNING  WARNING — unexpected but handled: missing CSV column
ERROR    ERROR   — something failed: json.JSONDecodeError caught
CRITICAL CRITICAL— serious failure: database unreachable


In [30]:
def safe_parse_llm_json(raw_response: str) -> dict:
    log.info(f"Parsing LLM response ({len(raw_response)}chars)")
    start = time.perf_counter()

    try:
        cleaned = raw_response.strip()
        if cleaned.startswith('```'):
            cleaned = cleaned.split('\n',1)[-1].rsplit('```',1)[0].strip()
            log.debug('Stripped Markdown fences from response')

        parsed = json.loads(cleaned)
        log.debug(f"Parsed Successfully. Keys: {list(parsed.keys())}")
        return parsed

    except json.JSONDecodeError as e:
        log.error(f"JSON parse failed at line {e.lineno} col {e.colno}: {e.msg}")
        log.error(f"Raw response was: {raw_response[:100]}")
        return {}
    finally:
        elapsed_ms = round((time.perf_counter() - start) * 1000,2)
        log.info(f"Parse finished in {elapsed_ms}ms")

#Demo
print("\n--- Test 1: valid JSON ---")
result = safe_parse_llm_json('{"category:" "TRACK_ORDER", "confidence": "high"}')
print(f"Result:  {result}")

print("\n--- Test 2: fenced JSON ---")
result2 = safe_parse_llm_json('```json\n{"intent": "CANCEL"}\n```')
print(f"Result: {result2}")

print("\n--- Test 3: invalid JSON ---")
result3 = safe_parse_llm_json('{bad json here}')
print(f"Result: {result3}")

INFO     Parsing LLM response (49chars)
ERROR    JSON parse failed at line 1 col 14: Expecting ':' delimiter
ERROR    Raw response was: {"category:" "TRACK_ORDER", "confidence": "high"}
INFO     Parse finished in 15.65ms
INFO     Parsing LLM response (32chars)
DEBUG    Stripped Markdown fences from response
DEBUG    Parsed Successfully. Keys: ['intent']
INFO     Parse finished in 5.66ms
INFO     Parsing LLM response (15chars)
ERROR    JSON parse failed at line 1 col 2: Expecting property name enclosed in double quotes
ERROR    Raw response was: {bad json here}
INFO     Parse finished in 7.37ms



--- Test 1: valid JSON ---
Result:  {}

--- Test 2: fenced JSON ---
Result: {'intent': 'CANCEL'}

--- Test 3: invalid JSON ---
Result: {}


In [8]:
def parse_customer_csv_row(row : dict) -> dict:
    try:
        return {
            'customer_id'   : int(row['customer_id']),
            'name'          : f"{row['first_name']}  {row['last_name']}".strip(),
            'email'         : row['email'].lower().strip(),
            'city'          : row['city'],
        }

    except KeyError as e:
        log.warning(f"Missing column in customer row: {e}. Row:{row}")
        return {}
    except ValueError as e:
        log.warning(f"Type conversion failed for customer row: {e}. Row: {row}")
        return  {}

# Demo
print("--- Valid row ---")
valid_row = {'customer_id': '1001', 'first_name': 'Yogi', 'last_name': 'Reddy', 'email': 'yogireddy@gmail.com', 'city': 'Bengaluru'}
print(parse_customer_csv_row(valid_row))

print("\n--- Missing column (KeyError) ---")
missing_col = {'customer_id': '1002', 'first_name': "srinu"}
print(parse_customer_csv_row(missing_col))

print("\n--- Bad type (ValueError) ---")
bad_type = {'customer_id': 'N/A', 'first_name': 'rohith', 'last_name': "sharma", 'email': 'rohithsharma45@gmail.com', "city" : "mumbai"}
print(parse_customer_csv_row(bad_type))

--- Valid row ---
{'customer_id': 1001, 'name': 'Yogi  Reddy', 'email': 'yogireddy@gmail.com', 'city': 'Bengaluru'}

--- Missing column (KeyError) ---


NameError: name 'log' is not defined

In [9]:
def call_llm_api_with_retry(
        prompt: str,
        model: str = 'gpt-4o',
        max_retries: int = 3,
) -> str:

    attempt     = 0
    late_error  = None

    while attempt < max_retries:
        try:
            log.info(f"LLM call attempt {attempt + 1}/{max_retries} | model = {model}")
            response = _simulate_llm_call(prompt, attempt)
            log.info(f"Succeded on attempt {attempt + 1}")
            return response
        except PermissionError as e:
            log.error(f"Auth failed - check your API key: {e}")
            raise

        except (ConnectionError, TimeoutError) as e:
            wait = 0.1 * (2 ** attempt)
            log.warning(f"Attempt {attempt + 1} failed ({type(e).__name__}). Waiting {wait:.1f}s")
            time.sleep(wait)
            last_error = e

        except json.JSONDecodeError as e:
            log.warning(f"LLM returned invalid JSON. Retrying with stricter instrucion.")
            prompt = prompt + '\n\nIMPORTANT: Respond only with vaild JSON.'
            last_error = e

        attempt +=1

    log.error(f"All {max_retries} attempts failed. Last error: {last_error}")
    raise RuntimeError(f"LLM call failed after {max_retries} attempts") from last_error

In [11]:
def simulate_llm_call(prompt: str, attempt: int) -> str:
    if attempt == 0:
        raise ConnectionError('HTTP 429: Too many Requests')
    elif attempt == 1:
        raise  TimeoutError('Request timed out after 30s')
    else:
        return '{"category": "TRACK_ORDER", "confidence": "high"}'

#Demo
print("----Retry Demo----")
try:
    response = call_llm_api_with_retry("Where is order #3042?", max_retries=3)
    print(f"Final Response: {response}")
except RuntimeError as e:
    print(f"All retries exhausted: {e}")

----Retry Demo----


NameError: name 'json' is not defined

In [16]:
import logging
logging.basicConfig(
    level=logging.WARNING,
    format= '%(levelname)-8s %(message)s',
    force= True,
)
log = logging.getLogger(__name__)

log.debug(  'DEBUG      - NOT shown (below WARNING level)')
log.info(   'INFO       - NOT Shown (below WARNING level)')
log.warning('WARNING    - shown')
log.error(  'ERROR      - shown')

print()
logging.basicConfig(level=logging.DEBUG, format='%(levelname)-8s %(message)s', force=True)
log = logging.getLogger(__name__)

WARNING  WARNING    - shown
ERROR    ERROR      - shown
